由於訓練是在 Colab GPU 環境執行，請確認 Google Drive 中的專案目錄（`Pick Up Pan Test/`）包含 `datasets.zip`（內含 `datasets/so101_pick_stylus` 的 `data/`、`videos/`、`meta/`）。



> **架構說明（ACT: Action Chunking with Transformers）**：本 Notebook 使用 LeRobot 官方 `ACTPolicy`（ResNet18 視覺骨幹 + CVAE + Transformer Encoder/Decoder），從頭訓練雙相機 SO-101 夾取策略，預測未來 `CHUNK = 50` 步 Action Chunk。

> **斷點續訓與自動存檔說明**：本 Notebook 保留完整斷點續訓機制（包含模型權重、前後處理器、Optimizer、LR Scheduler、Loss 紀錄與隨機種子），預設每 **500 steps** 透過本地暫存與原子同步更新 Google Drive 的 `output_act/checkpoints/latest`（與 SmolVLA / FC / CNN 目錄隔離不覆蓋）。

# 環境掛載與套件安裝
掛載 Google Drive 並安裝 `lerobot[dataset]` 與 `av`（`ACTPolicy` 內建於 LeRobot 核心與 torchvision，不需下載 SmolVLM2 大型權重）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "lerobot[dataset]==0.7.0" av
try:
    import lerobot.utils.import_utils as _iu
    _iu._require_package_cache.clear()
except Exception:
    pass

In [ ]:
import glob
import json
import math
import os
import random
import shutil
import time

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lerobot.configs.policies import PreTrainedConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.factory import make_policy, make_pre_post_processors

if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive 尚未掛載，請先執行上方 drive.mount('/content/drive') Cell！")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch:", torch.__version__)

# 載入實體 SO-101 雙相機資料集、建立 ACT 模型並偵測斷點
1. **自動適配特徵與統計量**：透過 `ACTConfig(chunk_size=CHUNK, n_action_steps=CHUNK, device=device)` 搭配 `make_policy(cfg, ds_meta=ds.meta)` 與 `make_pre_post_processors(policy.config, dataset_stats=ds.meta.stats)`，自動依資料集建立 `observation.state`、`observation.images.top`、`observation.images.front` 與正規化統計量。
2. **視覺骨幹與 Transformer**：預設使用 `resnet18` 預訓練視覺骨幹，結合 CVAE 與 Transformer 編解碼器輸出 `(CHUNK, 6)` 動作序列。
3. **自動偵測斷點**：若 `output_act/checkpoints/` 已有完整 checkpoint（`latest` 或 `act_pick_step_*`），會自動載入進度最高的權重與訓練狀態。

In [ ]:
if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive 尚未掛載，請先執行上方 drive.mount('/content/drive') Cell！")

PROJECT_DIR = "/content/drive/MyDrive/學校/交大/01_課程/04_Physical AI/Pick Up Pan Test"
DS_ZIP = os.path.join(PROJECT_DIR, "datasets.zip")
REPO_ID = "local/so101_pick_stylus"
DS_ROOT = "/content/datasets/so101_pick_stylus"

DRIVE_CHECKPOINT_ROOT = os.path.join(PROJECT_DIR, "output_act/checkpoints")
LOCAL_CHECKPOINT_ROOT = "/content/output_act/checkpoints"
LATEST_DIR = os.path.join(DRIVE_CHECKPOINT_ROOT, "latest")
os.makedirs(DRIVE_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(LOCAL_CHECKPOINT_ROOT, exist_ok=True)
AUTO_RESUME = True  # 若要從頭重訓，請改為 False 或清空 output_act/checkpoints

REQUIRED_CKPT_FILES = (
    "training_state.pt",
    "config.json",
    "model.safetensors",
    "policy_preprocessor.json",
    "policy_postprocessor.json",
)

if not os.path.isdir(DS_ROOT):
    import zipfile
    if not os.path.isfile(DS_ZIP):
        raise FileNotFoundError(f"找不到資料集壓縮檔：{DS_ZIP}")
    print(f"Extracting {DS_ZIP} to /content ...")
    with zipfile.ZipFile(DS_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extraction complete:", DS_ROOT)

def verify_checkpoint_dir(ckpt_dir):
    """確認 checkpoint 目錄內必要檔案與 processor 狀態檔皆存在且大小不為 0。"""
    if not os.path.isdir(ckpt_dir):
        return False
    for fname in REQUIRED_CKPT_FILES:
        fpath = os.path.join(ckpt_dir, fname)
        if not os.path.isfile(fpath) or os.path.getsize(fpath) == 0:
            return False
    for cfg_name in ("policy_preprocessor.json", "policy_postprocessor.json"):
        try:
            with open(os.path.join(ckpt_dir, cfg_name), "r", encoding="utf-8") as f:
                proc_cfg = json.load(f)
            for step_cfg in proc_cfg.get("steps", []):
                state_file = step_cfg.get("state_file")
                if state_file:
                    s_path = os.path.join(ckpt_dir, state_file)
                    if not os.path.isfile(s_path) or os.path.getsize(s_path) == 0:
                        return False
        except Exception:
            return False
    return True

def recover_atomic_checkpoints(checkpoint_root):
    """若前次原子寫入中途被強制終止，清理暫存目錄並還原備份。"""
    if not os.path.isdir(checkpoint_root):
        return
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if name == "_ckpt_staging" or (name.startswith("_") and (name.endswith("_tmp") or "_tmp_" in name)):
            shutil.rmtree(full_path, ignore_errors=True)
    for name in sorted(os.listdir(checkpoint_root), reverse=True):
        if name.startswith("_") and (name.endswith("_old") or "_old_" in name):
            orig_name = name[1:].rsplit("_old", 1)[0]
            old_dir = os.path.join(checkpoint_root, name)
            orig_dir = os.path.join(checkpoint_root, orig_name)
            if not os.path.exists(orig_dir):
                os.rename(old_dir, orig_dir)
            else:
                shutil.rmtree(old_dir, ignore_errors=True)

def find_latest_checkpoint(*checkpoint_roots):
    """掃描所有完整 checkpoint，回傳 step 數最高的 (目錄路徑, training_state dict)。"""
    best_dir, best_ckpt, best_step = None, None, -1
    for root in checkpoint_roots:
        if not os.path.isdir(root):
            continue
        for name in sorted(os.listdir(root)):
            if name.startswith("_"):
                continue
            cand_dir = os.path.join(root, name)
            if not verify_checkpoint_dir(cand_dir):
                continue
            try:
                ckpt = torch.load(os.path.join(cand_dir, "training_state.pt"), map_location="cpu", weights_only=False)
                ckpt_step = int(ckpt.get("step", -1))
                if ckpt_step > best_step or (ckpt_step == best_step and name == "latest"):
                    best_step = ckpt_step
                    best_dir = cand_dir
                    best_ckpt = ckpt
            except Exception as e:
                print(f"Warning: skipping corrupted checkpoint {cand_dir}: {e}")
    return best_dir, best_ckpt

recover_atomic_checkpoints(DRIVE_CHECKPOINT_ROOT)
recover_atomic_checkpoints(LOCAL_CHECKPOINT_ROOT)

ds_meta = LeRobotDatasetMetadata(REPO_ID, root=DS_ROOT)
FPS = ds_meta.fps
CHUNK = 50
delta_timestamps = {"action": [i / FPS for i in range(CHUNK)]}
ds = LeRobotDataset(REPO_ID, root=DS_ROOT, delta_timestamps=delta_timestamps, video_backend="pyav")
print(f"Loaded dataset: {ds.num_episodes} episodes, {ds.num_frames} frames @ {FPS} FPS")
print("Dataset cameras:", ds.meta.camera_keys)

resume_dir, resume_ckpt = (
    find_latest_checkpoint(DRIVE_CHECKPOINT_ROOT, LOCAL_CHECKPOINT_ROOT)
    if AUTO_RESUME else (None, None)
)

if resume_dir is not None:
    print(f"--> Resuming ACT from checkpoint: {resume_dir} (step {resume_ckpt['step']})")
    cfg = PreTrainedConfig.from_pretrained(resume_dir)
    cfg.pretrained_path = resume_dir
    cfg.device = device
    if hasattr(cfg, "chunk_size") and cfg.chunk_size != CHUNK:
        raise ValueError(f"Checkpoint chunk_size ({cfg.chunk_size}) 與 CHUNK ({CHUNK}) 不一致，請調整 CHUNK={cfg.chunk_size} 或設 AUTO_RESUME=False。")
    policy = make_policy(cfg, ds_meta=ds.meta)
    preprocess, postprocess = make_pre_post_processors(
        policy.config,
        pretrained_path=resume_dir,
        preprocessor_overrides={"device_processor": {"device": device}},
    )
else:
    print("--> Starting fresh ACT training from scratch (ResNet18 + Transformer CVAE)")
    cfg = ACTConfig(
        chunk_size=CHUNK,
        n_action_steps=CHUNK,
        device=device,
    )
    policy = make_policy(cfg, ds_meta=ds.meta)
    preprocess, postprocess = make_pre_post_processors(
        policy.config,
        dataset_stats=ds.meta.stats,
    )

print("Policy input features:", list(policy.config.input_features.keys()))
print("ACT ready:", round(sum(p.numel() for p in policy.parameters()) / 1e6, 2), "M params")

# 執行 ACT 訓練與雲端斷點儲存（支援自動同步至 Drive 並結束工作階段）
- 使用 `ACTConfig` 預設的 AdamW（區分 Transformer 與 ResNet Backbone 學習率，預設 `LR = 1e-5`），搭配 Linear Warmup + Cosine Decay LR Scheduler 與梯度裁剪。
- 先寫入本地暫存 `/content/output_act/checkpoints/_ckpt_staging` 並驗證，再原子同步至 Google Drive `output_act/checkpoints/latest` 與里程碑 `act_pick_step_{step}`。
- 訓練完成後會繪製 Loss 曲線、執行單筆 `select_action` 推論檢查，並自動 `drive.flush_and_unmount()` 與中斷 Colab 連線。

In [ ]:
BATCH_SIZE = 16                         # 若顯存充足可調高至 32
NUM_WORKERS = 0                         # 容器 /dev/shm 預設較小，設為 0 避免 Bus error（若 shm 足夠可改 2~4）
STEPS = 15000                           # 總訓練步數
SAVE_LATEST_FREQ = 500                  # 每 500 steps 原子更新 latest，降低單次中斷損失
MILESTONE_FREQ = 5000                   # 每 5,000 steps 保留一個里程碑 checkpoint
MAX_SESSION_MINUTES = None              # 不限制單次執行時間，直到全部 STEPS 訓練完成（若需限時可改為分鐘數）
LR = 1e-5                               # ACT 論文標準學習率（backbone 與 transformer）
WARMUP_STEPS = 500
AUTO_DISCONNECT_ON_COMPLETE = True      # 訓練完成並確認雲端存檔後，自動 flush Drive 並中斷 Colab 連線
AUTO_DISCONNECT_ON_SESSION_LIMIT = True # 達單次時間上限存檔後，自動 flush Drive 並中斷 Colab 連線

opt_cfg = policy.config.get_optimizer_preset()
opt_cfg.lr = LR
if hasattr(policy.config, "optimizer_lr"):
    policy.config.optimizer_lr = LR
if hasattr(policy.config, "optimizer_lr_backbone"):
    policy.config.optimizer_lr_backbone = LR
grad_clip_norm = getattr(opt_cfg, "grad_clip_norm", 10.0)
optimizer = opt_cfg.build(policy.get_optim_params())

sched_cfg = policy.config.get_scheduler_preset()
if sched_cfg is not None:
    if hasattr(sched_cfg, "peak_lr"):
        sched_cfg.peak_lr = LR
    scheduler = sched_cfg.build(optimizer, num_training_steps=STEPS)
else:
    def lr_lambda(current_step):
        if current_step < WARMUP_STEPS:
            return float(current_step + 1) / float(max(1, WARMUP_STEPS))
        progress = float(current_step - WARMUP_STEPS) / float(max(1, STEPS - WARMUP_STEPS))
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

start_step = 0
losses = []

if resume_ckpt is not None:
    start_step = int(resume_ckpt["step"])
    losses = list(resume_ckpt.get("losses", []))
    optimizer.load_state_dict(resume_ckpt["optimizer"])
    scheduler.load_state_dict(resume_ckpt["scheduler"])
    rng_state = resume_ckpt.get("rng_state")
    if rng_state is not None:
        random.setstate(rng_state["python"])
        np.random.set_state(rng_state["numpy"])
        torch.set_rng_state(rng_state["torch"])
        if torch.cuda.is_available() and rng_state.get("cuda") is not None:
            torch.cuda.set_rng_state_all(rng_state["cuda"])
    print(f"Restored training state at step {start_step}/{STEPS} (loaded {len(losses)} loss records)")

def sync_dir_atomically(src_dir, dst_dir):
    """以暫存目錄 + 原子更名方式將本地完整的 checkpoint 同步到雲端目標目錄。"""
    parent_dir = os.path.dirname(dst_dir)
    base_name = os.path.basename(dst_dir)
    tmp_dir = os.path.join(parent_dir, f"_{base_name}_tmp")
    old_dir = os.path.join(parent_dir, f"_{base_name}_old")
    os.makedirs(parent_dir, exist_ok=True)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    shutil.rmtree(old_dir, ignore_errors=True)
    if os.path.exists(tmp_dir):
        tmp_dir = os.path.join(parent_dir, f"_{base_name}_tmp_{int(time.time() * 1000)}")
    if os.path.exists(old_dir):
        old_dir = os.path.join(parent_dir, f"_{base_name}_old_{int(time.time() * 1000)}")
    shutil.copytree(src_dir, tmp_dir, dirs_exist_ok=True)
    os.sync()
    if os.path.exists(dst_dir):
        os.rename(dst_dir, old_dir)
    os.rename(tmp_dir, dst_dir)
    shutil.rmtree(old_dir, ignore_errors=True)
    os.sync()

def save_checkpoint(target_dirs, current_step):
    """先寫入本地 SSD 暫存並驗證，再原子同步至一或多個雲端目標目錄。"""
    if isinstance(target_dirs, str):
        target_dirs = [target_dirs]
    staging_dir = os.path.join(LOCAL_CHECKPOINT_ROOT, "_ckpt_staging")
    shutil.rmtree(staging_dir, ignore_errors=True)
    os.makedirs(staging_dir, exist_ok=True)

    state_payload = {
        "step": current_step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "losses": losses,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
    }
    try:
        policy.save_pretrained(staging_dir)
        preprocess.save_pretrained(staging_dir)
        postprocess.save_pretrained(staging_dir)
        torch.save(state_payload, os.path.join(staging_dir, "training_state.pt"))
        if not verify_checkpoint_dir(staging_dir):
            raise RuntimeError(f"本地暫存 checkpoint 驗證失敗：{staging_dir}")

        for dst_dir in target_dirs:
            sync_dir_atomically(staging_dir, dst_dir)
            if not verify_checkpoint_dir(dst_dir):
                raise RuntimeError(f"雲端 checkpoint 驗證失敗：{dst_dir}")
    finally:
        shutil.rmtree(staging_dir, ignore_errors=True)

def build_trimmed_subset(
    dataset,
    ds_root,
    move_thresh=3.0,
    keep_lead_frames=10,
    keep_tail_frames=10,
    trim_head=True,
    trim_tail=True,
    drop_longest_n=10,
    exclude_episodes=None,
):
    """排除時間最長／指定 episode，並自動裁切每條 episode 開頭與結尾的靜止等待幀。"""
    files = sorted(glob.glob(f"{ds_root}/data/*/*.parquet"))
    if not files:
        raise FileNotFoundError(f"在 {ds_root}/data/*/*.parquet 找不到任何 parquet 資料檔。")
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    sort_cols = [c for c in ("episode_index", "frame_index", "index") if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    states = np.stack(df["observation.state"].values)
    eps = df["episode_index"].values
    indices = df["index"].values
    timestamps = df["timestamp"].values if "timestamp" in df.columns else None

    unique_eps = np.unique(eps)
    ep_masks = {int(ep): (eps == ep) for ep in unique_eps}

    excluded_set = {int(e) for e in (exclude_episodes or ())}
    remaining_eps = [int(ep) for ep in unique_eps if int(ep) not in excluded_set]

    longest_dropped = []
    if drop_longest_n > 0:
        if len(remaining_eps) <= drop_longest_n:
            raise ValueError(
                f"可用 episode 數 ({len(remaining_eps)}) 不足，無法排除最長的 {drop_longest_n} 筆。"
            )
        ep_durations = []
        for ep in remaining_eps:
            m = ep_masks[ep]
            n_frames = int(np.sum(m))
            if timestamps is not None and n_frames > 1:
                t_start = float(np.asarray(timestamps[m][0]).item())
                t_end = float(np.asarray(timestamps[m][-1]).item())
                dur = t_end - t_start
            else:
                dur = float(n_frames)
            ep_durations.append((dur, n_frames, ep))
        ep_durations.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
        for dur, n_frames, ep in ep_durations[:drop_longest_n]:
            excluded_set.add(ep)
            longest_dropped.append((ep, n_frames, dur))

    valid_indices = []
    dropped_ep_frames = 0
    trimmed_head_frames = 0
    trimmed_tail_frames = 0

    for ep in unique_eps:
        ep_int = int(ep)
        m = ep_masks[ep_int]
        ep_idx = indices[m]
        if ep_int in excluded_set:
            dropped_ep_frames += len(ep_idx)
            continue

        ep_states = states[m]
        n_ep = len(ep_states)

        if trim_head:
            head_diffs = np.max(np.abs(ep_states - ep_states[0]), axis=1)
            head_moved = np.where(head_diffs > move_thresh)[0]
            start_offset = max(0, int(head_moved[0]) - keep_lead_frames) if len(head_moved) > 0 else 0
        else:
            start_offset = 0

        if trim_tail:
            tail_diffs = np.max(np.abs(ep_states - ep_states[-1]), axis=1)
            tail_moved = np.where(tail_diffs > move_thresh)[0]
            end_offset = (
                min(n_ep, int(tail_moved[-1]) + 1 + keep_tail_frames)
                if len(tail_moved) > 0
                else n_ep
            )
        else:
            end_offset = n_ep

        if end_offset <= start_offset:
            start_offset = 0
            end_offset = n_ep

        trimmed_head_frames += start_offset
        trimmed_tail_frames += n_ep - end_offset
        valid_indices.extend(ep_idx[start_offset:end_offset].tolist())

    if longest_dropped:
        dropped_desc = ", ".join(
            f"ep{ep}({n}f/{dur:.1f}s)" if timestamps is not None else f"ep{ep}({n}f)"
            for ep, n, dur in longest_dropped
        )
        print(f"Excluded longest {len(longest_dropped)} episodes: {dropped_desc}")
    if len(excluded_set) > len(longest_dropped):
        manual_eps = sorted(excluded_set - {ep for ep, _, _ in longest_dropped})
        print(f"Excluded manual episodes: {manual_eps}")

    removed = len(df) - len(valid_indices)
    print(
        f"Dataset filter summary: {len(df)} -> {len(valid_indices)} frames "
        f"(kept {len(valid_indices) / len(df) * 100:.1f}% | "
        f"dropped_eps={dropped_ep_frames}, trimmed_head={trimmed_head_frames}, "
        f"trimmed_tail={trimmed_tail_frames}, total_removed={removed})"
    )
    return torch.utils.data.Subset(dataset, valid_indices)

exit_reason = None
last_completed_step = start_step

if start_step >= STEPS:
    print(f"Training already completed ({start_step}/{STEPS} steps). Skipping loop.")
    exit_reason = "completed"
else:
    train_ds = build_trimmed_subset(
        ds,
        DS_ROOT,
        move_thresh=3.0,
        keep_lead_frames=10,
        keep_tail_frames=10,
        trim_head=True,
        trim_tail=True,
        drop_longest_n=10,
    )
    if len(train_ds) < BATCH_SIZE:
        raise ValueError(f"可用訓練樣本數 ({len(train_ds)}) 小於 BATCH_SIZE ({BATCH_SIZE})。")
    loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda" and NUM_WORKERS > 0),
    )
    policy.train()
    it = iter(loader)
    session_start_t = time.time()

    try:
        for step in range(start_step + 1, STEPS + 1):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(loader)
                batch = next(it)

            batch = preprocess(batch)
            out = policy(batch)
            loss = out[0] if isinstance(out, (tuple, list)) else out["loss"]

            optimizer.zero_grad()
            loss.backward()
            if grad_clip_norm and grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(policy.parameters(), grad_clip_norm)
            optimizer.step()
            scheduler.step()

            loss_val = loss.detach().item()
            losses.append(loss_val)
            last_completed_step = step
            elapsed_min = (time.time() - session_start_t) / 60.0

            if step % 200 == 0 or step == start_step + 1:
                cur_lr = optimizer.param_groups[0]["lr"]
                print(f"step {step:5d}/{STEPS} | loss {loss_val:.4f} | lr {cur_lr:.2e} | elapsed {elapsed_min:.1f}m")

            reach_limit = (
                MAX_SESSION_MINUTES is not None
                and elapsed_min >= MAX_SESSION_MINUTES
                and step < STEPS
            )
            save_targets = []
            if step % SAVE_LATEST_FREQ == 0 or step == STEPS or reach_limit:
                save_targets.append(LATEST_DIR)
            if step % MILESTONE_FREQ == 0 or step == STEPS:
                save_targets.append(os.path.join(DRIVE_CHECKPOINT_ROOT, f"act_pick_step_{step}"))

            if save_targets:
                save_checkpoint(save_targets, step)
                for t_dir in save_targets:
                    print(f"--> Checkpoint synced & verified at step {step}: {t_dir}")

            if reach_limit:
                exit_reason = "session_limit"
                print(
                    f"\n[Session Limit] Reached {elapsed_min:.1f}m (>= {MAX_SESSION_MINUTES}m). "
                    f"Saved progress at step {step}/{STEPS} to {LATEST_DIR}."
                )
                break
        else:
            exit_reason = "completed"
    except KeyboardInterrupt:
        exit_reason = "interrupted"
        if last_completed_step > start_step:
            save_checkpoint([LATEST_DIR], last_completed_step)
            print(f"\n[Interrupted] Saved & verified progress at step {last_completed_step}/{STEPS} to {LATEST_DIR}.")
    except Exception:
        exit_reason = "error"
        if last_completed_step > start_step:
            try:
                save_checkpoint([LATEST_DIR], last_completed_step)
                print(f"\n[Error Recovery] Saved progress at step {last_completed_step}/{STEPS} to {LATEST_DIR}.")
            except Exception as save_err:
                print(f"\n[Error Recovery] Emergency save failed: {save_err}")
        raise

if losses:
    plt.figure(figsize=(8, 3))
    plt.plot(range(1, len(losses) + 1), losses, alpha=0.7)
    plt.xlabel("step")
    plt.ylabel("ACT loss (L1 + KL)")
    plt.title("ACT Training Loss (SO-101 Pick Stylus)")
    if all(v > 0 for v in losses):
        plt.yscale("log")
    plt.grid(True)
    plt.show()

if exit_reason in ("completed", "session_limit"):
    # 推論檢查
    policy.eval()
    policy.reset()
    sample = ds[0]
    obs_batch = {
        k: v.unsqueeze(0)
        for k, v in sample.items()
        if k.startswith("observation.") and isinstance(v, torch.Tensor)
    }
    with torch.no_grad():
        pred_action = postprocess(policy.select_action(preprocess(obs_batch)))
    print("Sanity check - predicted action shape:", tuple(pred_action.shape))
    print("Sanity check - first predicted step:", pred_action.squeeze(0).detach().cpu().numpy().round(3))

    final_milestone = os.path.join(DRIVE_CHECKPOINT_ROOT, f"act_pick_step_{STEPS}")
    verify_target = (
        final_milestone if (exit_reason == "completed" and verify_checkpoint_dir(final_milestone))
        else LATEST_DIR
    )
    if not verify_checkpoint_dir(verify_target):
        raise RuntimeError(f"雲端最終檢查失敗，找不到完整模型檔案：{verify_target}")

    weights_mb = os.path.getsize(os.path.join(verify_target, "model.safetensors")) / (1024 * 1024)
    print(f"[Cloud Verified] 模型已完整儲存至 Google Drive：{verify_target} (model.safetensors: {weights_mb:.1f} MB)")

    should_disconnect = (
        (exit_reason == "completed" and AUTO_DISCONNECT_ON_COMPLETE)
        or (exit_reason == "session_limit" and AUTO_DISCONNECT_ON_SESSION_LIMIT)
    )
    if should_disconnect:
        os.sync()
        try:
            from google.colab import drive, runtime
            print("[Cloud Sync] Flushing Google Drive writes before disconnecting...")
            drive.flush_and_unmount()
            print("[Runtime] Disconnecting Colab runtime now...")
            runtime.unassign()
        except ImportError:
            print("[Runtime] 非 Colab 環境，略過 flush_and_unmount / runtime.unassign。")